In [1]:
import torch, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tabpfn_extensions import TabPFNClassifier           # 记得是 extensions 里的
from models.tabular_encoder import tabular_encoder_classifier

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
csv_path = rf"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
label_col  = "Group"
classes    = ["SMCI", "PMCI"]
# feature_cols = ["MMSE"]
n_fold     = 5
dropna     = False
test_size  = 0.2
start_col  = 4 

▶ Using device: cuda:0


In [2]:
# ---------------- 生成嵌入 -----------------
train_df, test_df = tabular_encoder_classifier(
    csv_path   = csv_path,
    label_col  = label_col,
    classes    = classes,
    # feature_cols = feature_cols,
    n_fold     = n_fold,
    dropna     = dropna,
    test_size  = test_size,
    start_col  = start_col
)

# ---------- 拆分 X / y ----------
X_train = train_df.drop(columns=["label"]).values       # shape (N_tr, 192)
y_train = train_df["label"].values.astype("int64")

X_test  = test_df.drop(columns=["label"]).values
y_test  = test_df["label"].values.astype("int64")

print("Train:", X_train.shape, "Test:", X_test.shape)

✓ train_emb (383, 192) · test_emb (96, 192)
已保存到: train_embeddings.csv  /  test_embeddings.csv
Train: (383, 192) Test: (96, 192)


In [ ]:
# ---------- 3. 用 TabPFNClassifier 做最终分类 ----------
clf = TabPFNClassifier(device=DEVICE)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]                # P(Positive)
y_pred = (y_prob >= 0.5).astype("int64")                # 默认 0.5 阈值

# ---------- 4. 评估 ----------
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nAccuracy = {acc:.4f} · AUC = {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=classes))



Accuracy = 0.8021 · AUC = 0.8813

              precision    recall  f1-score   support

        SMCI       0.84      0.88      0.85        64
        PMCI       0.72      0.66      0.69        32

    accuracy                           0.80        96
   macro avg       0.78      0.77      0.77        96
weighted avg       0.80      0.80      0.80        96



In [4]:
def quick_eval_from_saved(train_csv="train_embeddings.csv", test_csv="test_embeddings.csv"):
    """
    读取带标签的嵌入 CSV，使用 **最简 SVM** (线性核) 做一次快速评估。
    """
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)

    y_tr, X_tr = tr["label"].values, tr.drop(columns="label").values
    y_te, X_te = te["label"].values, te.drop(columns="label").values

    # 使用线性核 SVM, 外加标准化, 这是最简易且常用的组合
    clf = make_pipeline(StandardScaler(), SVC(kernel="linear"))
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"[quick eval · SVM-linear] Accuracy on {test_csv}: {acc:.4f}")
    return acc

quick_eval_from_saved()

[quick eval · SVM-linear] Accuracy on test_embeddings.csv: 0.7396


0.7395833333333334

In [5]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

def quick_eval_from_saved(train_csv="train_embeddings.csv",
                          test_csv="test_embeddings.csv"):
    """
    读取嵌入 CSV，用线性核 SVM 评估。
    同时输出 Accuracy 和 AUC。
    """
    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)

    y_tr, X_tr = tr["label"].values, tr.drop(columns="label").values
    y_te, X_te = te["label"].values, te.drop(columns="label").values

    # probability=True 以便后续计算 AUC
    clf = make_pipeline(StandardScaler(),
                        SVC(kernel="linear", probability=True, random_state=42))
    clf.fit(X_tr, y_tr)

    y_pred  = clf.predict(X_te)
    acc     = accuracy_score(y_te, y_pred)

    # 计算 AUC: 二分类直接取正类概率，多分类用 OVR/OVO
    if len(np.unique(y_tr)) == 2:
        y_score = clf.predict_proba(X_te)[:, 1]
        auc     = roc_auc_score(y_te, y_score)
    else:
        y_score = clf.predict_proba(X_te)
        auc     = roc_auc_score(y_te, y_score,
                                multi_class="ovr", average="weighted")

    print(f"[quick eval · SVM-linear] Accuracy: {acc:.4f} | AUC: {auc:.4f}")
    return acc, auc

# 调用
quick_eval_from_saved()


[quick eval · SVM-linear] Accuracy: 0.7396 | AUC: 0.8350


(0.7395833333333334, np.float64(0.8349609375))

In [8]:
# -*- coding: utf-8 -*-
"""
直接运行版：表格嵌入 -> TabPFNClassifier 训练评估 -> 快速MLP评估
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# -------- 1) 依赖导入 --------
from tabpfn_extensions import TabPFNClassifier           # 注意：来自 extensions
from models.tabular_encoder import tabular_encoder_classifier


# -------- 2) 快速评估函数（基于已保存CSV嵌入，MLP分类器）--------
def quick_eval_with_mlp(train_csv="train_embeddings.csv", test_csv="test_embeddings.csv",
                        hidden_dim=64, epochs=50, lr=1e-3, device="cpu"):
    """
    读取嵌入 CSV，用两层全连接的 MLP 分类头做快速评估。
    """
    # 读数据
    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)

    y_tr, X_tr = tr["label"].values.astype("int64"), tr.drop(columns="label").values.astype(np.float32)
    y_te, X_te = te["label"].values.astype("int64"), te.drop(columns="label").values.astype(np.float32)

    X_tr = torch.tensor(X_tr, device=device)
    y_tr = torch.tensor(y_tr, device=device)
    X_te = torch.tensor(X_te, device=device)
    y_te = torch.tensor(y_te, device=device)

    input_dim = X_tr.shape[1]
    num_classes = len(np.unique(y_tr.cpu().numpy()))

    # 定义两层全连接分类头
    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, num_classes)
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 训练
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tr)
        loss = criterion(outputs, y_tr)
        loss.backward()
        optimizer.step()

    # 评估
    model.eval()
    with torch.no_grad():
        logits = model(X_te)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits, dim=1).cpu().numpy()

    acc = accuracy_score(y_te.cpu().numpy(), preds)
    auc = roc_auc_score(y_te.cpu().numpy(), probs) if num_classes == 2 else float("nan")
    print(f"[quick eval · MLP] Accuracy = {acc:.4f} · AUC = {auc:.4f}")
    return acc, auc


# -------- 3) 主流程 --------
if __name__ == "__main__":
    # ======== 固定参数 ========
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    csv_path = r"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
    label_col = "Group"
    classes = ["SMCI", "PMCI"]
    n_fold = 5
    dropna = False
    test_size = 0.2
    start_col = 4
    threshold = 0.5

    # 是否保存嵌入 + 是否进行快速评估
    save_embeddings = True
    quick_eval = True
    train_embeddings_csv = "train_embeddings.csv"
    test_embeddings_csv = "test_embeddings.csv"

    # --- 3.1 生成嵌入 ---
    train_df, test_df = tabular_encoder_classifier(
        csv_path=csv_path,
        label_col=label_col,
        classes=classes,
        n_fold=n_fold,
        dropna=dropna,
        test_size=test_size,
        start_col=start_col
    )

    # 保存嵌入
    if save_embeddings:
        train_df.to_csv(train_embeddings_csv, index=False)
        test_df.to_csv(test_embeddings_csv, index=False)
        print(f"[info] 训练嵌入已保存至: {train_embeddings_csv}")
        print(f"[info] 测试嵌入已保存至:  {test_embeddings_csv}")

    # --- 3.2 拆分 X / y ---
    X_train = train_df.drop(columns=["label"]).values
    y_train = train_df["label"].values.astype("int64")

    X_test = test_df.drop(columns=["label"]).values
    y_test = test_df["label"].values.astype("int64")

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # --- 3.3 TabPFNClassifier 训练与预测 ---
    clf = TabPFNClassifier(device=DEVICE)
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]                  # P(Positive)
    y_pred = (y_prob >= threshold).astype("int64")            # 可调阈值

    # --- 3.4 评估 ---
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"\n[TabPFN] Accuracy = {acc:.4f} · AUC = {auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=classes))

    # --- 3.5 快速MLP评估 ---
    if quick_eval:
        quick_eval_with_mlp(train_embeddings_csv, test_embeddings_csv,
                            hidden_dim=64, epochs=50, lr=1e-3, device=DEVICE)



✓ train_emb (383, 192) · test_emb (96, 192)
已保存到: train_embeddings.csv  /  test_embeddings.csv
[info] 训练嵌入已保存至: train_embeddings.csv
[info] 测试嵌入已保存至:  test_embeddings.csv
Train: (383, 192) Test: (96, 192)

[TabPFN] Accuracy = 0.8021 · AUC = 0.8813
Confusion Matrix:
 [[56  8]
 [11 21]]

Classification Report:
              precision    recall  f1-score   support

        SMCI       0.84      0.88      0.85        64
        PMCI       0.72      0.66      0.69        32

    accuracy                           0.80        96
   macro avg       0.78      0.77      0.77        96
weighted avg       0.80      0.80      0.80        96

[quick eval · MLP] Accuracy = 0.8021 · AUC = 0.8882


In [10]:
# -*- coding: utf-8 -*-
"""
直接运行版：表格嵌入 -> TabPFNClassifier 训练评估 -> 快速MLP评估 -> SHAP特征重要性
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# -------- 1) 依赖导入 --------
from tabpfn_extensions import TabPFNClassifier           # 注意：来自 extensions
from models.tabular_encoder import tabular_encoder_classifier

# 可选：更友好的缺包提示
try:
    import shap
except Exception as e:
    shap = None
    print("[warn] 未安装 shap，将无法执行 SHAP 分析。请先运行: pip install shap")


# -------- 2) 快速评估（MLP分类头）+ SHAP 分析 --------
def quick_eval_with_mlp_and_shap(
    train_csv="train_embeddings.csv",
    test_csv="test_embeddings.csv",
    hidden_dim=64,
    epochs=50,
    lr=1e-3,
    device="cpu",
    shap_save_csv="shap_importance.csv",
    shap_topk_print=20,
):
    import warnings
    try:
        import shap
    except Exception as e:
        shap = None
        warnings.warn("未安装 shap，跳过 SHAP 分析。请运行: pip install shap")

    # 读数据
    tr = pd.read_csv(train_csv)
    te = pd.read_csv(test_csv)

    feature_names = [c for c in tr.columns if c != "label"]
    y_tr = tr["label"].values.astype("int64")
    X_tr = tr.drop(columns="label").values.astype(np.float32)
    y_te = te["label"].values.astype("int64")
    X_te = te.drop(columns="label").values.astype(np.float32)

    X_tr_t = torch.tensor(X_tr, device=device)
    y_tr_t = torch.tensor(y_tr, device=device)
    X_te_t = torch.tensor(X_te, device=device)
    y_te_t = torch.tensor(y_te, device=device)

    input_dim = X_tr_t.shape[1]
    num_classes = len(np.unique(y_tr))

    # 两层MLP分类头
    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, num_classes),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tr_t)
        loss = criterion(outputs, y_tr_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_te_t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_te, preds)
    auc = roc_auc_score(y_te, probs[:, 1]) if num_classes == 2 else float("nan")
    print(f"[quick eval · MLP] Accuracy = {acc:.4f} · AUC = {auc:.4f}")

    # ===== SHAP =====
    if shap is None:
        return acc, auc

    print("[info] 开始 SHAP 分析（DeepExplainer, PyTorch）...")
    # 背景样本不要太大，避免过慢
    bg_size = min(200, X_tr_t.shape[0])
    background = X_tr_t[:bg_size]

    explainer = shap.DeepExplainer(model, background)
    shap_values = explainer.shap_values(X_te_t)  # 可能是 list 或 ndarray，shape 各异

    n_samples = X_te_t.shape[0]
    n_features = X_te_t.shape[1]
    C = num_classes

    def _as_list_of_class_arrays(sv):
        """
        统一为 List[np.ndarray]，每个元素形状为 (n_samples, n_features)。
        兼容多种返回形态。
        """
        # list形式（标准多分类）：每个类一个 (N, F)
        if isinstance(sv, list):
            out = []
            for a in sv:
                a = np.asarray(a)
                # 某些实现会给 (F,) or (N,F)；做健壮处理
                if a.ndim == 1 and a.shape[0] == n_features:
                    a = np.tile(a[None, :], (n_samples, 1))
                elif a.ndim == 3 and a.shape[1] == 1 and a.shape[2] == n_features:
                    # (N,1,F) -> (N,F)
                    a = a[:, 0, :]
                out.append(a.reshape(n_samples, n_features))
            return out

        # ndarray 形式
        sv = np.asarray(sv)
        # (N, F) -> 单输出（二分类时常见）
        if sv.ndim == 2 and sv.shape == (n_samples, n_features):
            # 视作“正类”的解释；也可复制为两列以便对齐
            return [sv]

        # (N, C, F)
        if sv.ndim == 3 and sv.shape[0] == n_samples and sv.shape[1] == C and sv.shape[2] == n_features:
            return [sv[:, k, :] for k in range(C)]

        # (C, N, F)
        if sv.ndim == 3 and sv.shape[0] == C and sv.shape[1] == n_samples and sv.shape[2] == n_features:
            return [sv[k, :, :] for k in range(C)]

        # (N, F, C)
        if sv.ndim == 3 and sv.shape[0] == n_samples and sv.shape[1] == n_features and sv.shape[2] == C:
            return [sv[:, :, k] for k in range(C)]

        # 其他少见形状，尝试自动识别：找到哪个轴等于 n_features，哪个轴等于 n_samples
        axes = sv.shape
        try:
            idx_f = int(np.where(np.array(axes) == n_features)[0][0])
            idx_n = int(np.where(np.array(axes) == n_samples)[0][0])
            # 剩下的那个轴当作类别轴
            idx_c = [i for i in range(sv.ndim) if i not in (idx_f, idx_n)][0]
            # 调整为 (N, C, F)
            sv_perm = np.moveaxis(sv, (idx_n, idx_c, idx_f), (0, 1, 2))
            return [sv_perm[:, k, :] for k in range(sv_perm.shape[1])]
        except Exception as _:
            raise ValueError(f"无法解析 shap_values 的形状 {sv.shape} 与 (n_samples={n_samples}, n_features={n_features}, n_classes={C}) 对应关系。")

    class_wise = _as_list_of_class_arrays(shap_values)  # List[(N,F)]
    # 计算每类的 mean(|SHAP|) 列重要性
    class_imps = [np.mean(np.abs(a), axis=0) for a in class_wise]  # List[(F,)]

    # 组装 DataFrame
    data = {"feature": feature_names}
    for k, imp in enumerate(class_imps):
        data[f"importance_class_{k}"] = imp
    imp_df = pd.DataFrame(data)
    if len(class_imps) >= 2:
        imp_df["importance_mean"] = imp_df[[c for c in imp_df.columns if c.startswith("importance_class_")]].mean(axis=1)
        sort_col = "importance_mean"
    else:
        sort_col = "importance_class_0"

    imp_df_sorted = imp_df.sort_values(sort_col, ascending=False)
    print(f"\n[SHAP] Top-{shap_topk_print} 重要特征（基于 mean(|SHAP|)）:")
    print(imp_df_sorted.head(shap_topk_print).to_string(index=False))

    imp_df_sorted.to_csv(shap_save_csv, index=False, encoding="utf-8-sig")
    print(f"[info] SHAP 列重要性已保存至: {os.path.abspath(shap_save_csv)}")

    return acc, auc



# -------- 3) 主流程 --------
if __name__ == "__main__":
    # ======== 固定参数 ========
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    csv_path = r"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
    label_col = "Group"
    classes = ["SMCI", "PMCI"]
    n_fold = 5
    dropna = False
    test_size = 0.2
    start_col = 4
    threshold = 0.5

    # 是否保存嵌入 + 是否进行快速评估（含 SHAP）
    save_embeddings = True
    quick_eval = True
    train_embeddings_csv = "train_embeddings.csv"
    test_embeddings_csv = "test_embeddings.csv"

    # --- 3.1 生成嵌入 ---
    train_df, test_df = tabular_encoder_classifier(
        csv_path=csv_path,
        label_col=label_col,
        classes=classes,
        n_fold=n_fold,
        dropna=dropna,
        test_size=test_size,
        start_col=start_col
    )

    # 保存嵌入
    if save_embeddings:
        train_df.to_csv(train_embeddings_csv, index=False)
        test_df.to_csv(test_embeddings_csv, index=False)
        print(f"[info] 训练嵌入已保存至: {train_embeddings_csv}")
        print(f"[info] 测试嵌入已保存至:  {test_embeddings_csv}")

    # --- 3.2 拆分 X / y ---
    X_train = train_df.drop(columns=["label"]).values
    y_train = train_df["label"].values.astype("int64")

    X_test  = test_df.drop(columns=["label"]).values
    y_test  = test_df["label"].values.astype("int64")

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # --- 3.3 TabPFNClassifier 训练与预测 ---
    clf = TabPFNClassifier(device=DEVICE)
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]                  # P(Positive)
    y_pred = (y_prob >= threshold).astype("int64")            # 可调阈值

    # --- 3.4 评估 ---
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"\n[TabPFN] Accuracy = {acc:.4f} · AUC = {auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=classes))

    # --- 3.5 快速MLP评估 + SHAP 列重要性 ---
    if quick_eval:
        quick_eval_with_mlp_and_shap(
            train_embeddings_csv,
            test_embeddings_csv,
            hidden_dim=64,
            epochs=50,
            lr=1e-3,
            device=DEVICE,
            shap_save_csv="shap_importance.csv",
            shap_topk_print=20,
        )


✓ train_emb (383, 192) · test_emb (96, 192)
已保存到: train_embeddings.csv  /  test_embeddings.csv
[info] 训练嵌入已保存至: train_embeddings.csv
[info] 测试嵌入已保存至:  test_embeddings.csv
Train: (383, 192) Test: (96, 192)

[TabPFN] Accuracy = 0.8021 · AUC = 0.8813
Confusion Matrix:
 [[56  8]
 [11 21]]

Classification Report:
              precision    recall  f1-score   support

        SMCI       0.84      0.88      0.85        64
        PMCI       0.72      0.66      0.69        32

    accuracy                           0.80        96
   macro avg       0.78      0.77      0.77        96
weighted avg       0.80      0.80      0.80        96

[quick eval · MLP] Accuracy = 0.8021 · AUC = 0.8877
[info] 开始 SHAP 分析（DeepExplainer, PyTorch）...

[SHAP] Top-20 重要特征（基于 mean(|SHAP|)）:
feature  importance_class_0  importance_class_1  importance_mean
    185            0.062092            0.077862         0.069977
     26            0.038094            0.050729         0.044412
     60            0.034678      

In [3]:
# -*- coding: utf-8 -*-
"""
TabPFN + SHAP(shapiq) · 原始列层面解释（带可视化进度与阶段耗时 + 稳健修复）
依赖：
    pip install tabpfn shapiq pandas scikit-learn numpy torch tqdm
"""

import os
import numpy as np
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from tqdm import tqdm

# --- TabPFN 兼容导入（官方包优先，其次 extensions） ---
try:
    from tabpfn import TabPFNClassifier  # 官方
except Exception:
    from tabpfn_extensions import TabPFNClassifier  # 你的 extensions 版本

import shapiq  # pip install shapiq


# ===================== 用户参数 =====================
CSV_PATH    = r"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
LABEL_COL   = "Group"
CLASSES     = ["SMCI", "PMCI"]       # 目标二分类标签
START_COL   = 4                      # 原始特征起始列（含）
TEST_SIZE   = 0.2
RANDOM_SEED = 42

# SHAP 参数（先小一点，确认流程 OK 再调大）
EXPLAIN_MAX_EVAL = 64               # 从测试集中抽多少样本做解释
SHAP_BUDGET      = 64               # Shapley 预算（子集采样预算，越大越稳越慢）
TOPK_PRINT       = 30
OUT_CSV          = "tabpfn_shap_importance_raw.csv"
OUT_CSV_PARTIAL  = "tabpfn_shap_importance_raw_partial.csv"  # 增量保存文件
SAVE_EVERY       = 32                # 每处理多少样本增量保存一次

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def section(msg: str):
    print("\n" + "=" * 12 + f" {msg} " + "=" * 12, flush=True)


def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    t0 = perf_counter()

    # ===================== 读取&清洗 =====================
    section("数据读取与清洗")
    t = perf_counter()
    df = pd.read_csv(CSV_PATH)

    # 原始特征列
    raw_cols = list(df.columns[START_COL:])

    # 标签清洗：统一去空格
    labels_raw = df[LABEL_COL].astype(str).str.strip()

    # 映射
    mapping = {CLASSES[0]: 0, CLASSES[1]: 1}
    y_map = labels_raw.map(mapping)

    # 二次“大小写无关”兜底
    if y_map.isna().any():
        lower_map = {CLASSES[0].lower(): 0, CLASSES[1].lower(): 1}
        y_map2 = labels_raw.str.lower().map(lower_map)
        y_map = y_map.fillna(y_map2)

    # 未映射的提示并剔除
    if y_map.isna().any():
        unmapped_vals = labels_raw[y_map.isna()].value_counts()
        print("[warn] 发现未映射到的标签值（将被剔除）：")
        print(unmapped_vals.to_string())
        print("[hint] 如需包含它们，请调整 CLASSES 或预先过滤。")

    keep_mask = ~y_map.isna()
    dropped = (~keep_mask).sum()
    if dropped > 0:
        print(f"[info] 因标签未映射/缺失删除样本数：{dropped}")
    df = df.loc[keep_mask].reset_index(drop=True)
    y_all = y_map.loc[keep_mask].astype(int).values

    # 若只剩一个类别，直接报错提示
    if np.unique(y_all).size < 2:
        raise ValueError(f"仅剩单一类别：{np.unique(y_all)}。请检查 LABEL_COL/CLASSES 或过滤策略。")

    # 特征清洗：inf/-inf -> NaN -> 中位数插补
    X_df = df[raw_cols].copy()
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="median")
    X_all = imputer.fit_transform(X_df.values).astype(np.float32)

    # === 新增：过滤零方差特征（全局）===
    rng = np.ptp(X_all, axis=0)  # max-min
    nonconst_mask = rng > 1e-12  # 或用 X_all.std(axis=0) > 1e-12
    if nonconst_mask.sum() < len(nonconst_mask):
        removed = [c for c, keep in zip(raw_cols, nonconst_mask) if not keep]
        print(f"[info] 过滤零方差特征 {len(removed)} 列：{removed[:10]}{' ...' if len(removed)>10 else ''}")
    X_all = X_all[:, nonconst_mask]
    raw_cols = [c for c, keep in zip(raw_cols, nonconst_mask) if keep]

    # 断言无 NaN/Inf
    if not np.isfinite(X_all).all():
        raise ValueError("插补后仍存在 NaN/Inf，请检查数据。")

    print(f"[done] 用时 {(perf_counter()-t):.2f}s · 数据形状: X={X_all.shape}, y={y_all.shape[0]}")

    # ===================== 划分 =====================
    section("划分训练/测试集")
    t = perf_counter()
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
    )
    print(f"Train: {X_train.shape}, Test: {X_test.shape}, Pos rate(train)={y_train.mean():.3f}")
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # ===================== 训练 TabPFN =====================
    section("训练 TabPFN")
    t = perf_counter()
    model = TabPFNClassifier(device=DEVICE)
    model.fit(X_train, y_train)
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # 评估
    section("评估 TabPFN")
    t = perf_counter()
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"[TabPFN] ACC={acc:.4f}  AUC={auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASSES))
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # ===================== SHAP（原始列层面，带进度条） =====================
    section("SHAP（原始列 · TabPFNExplainer · 带进度）")
    t = perf_counter()
    explainer = shapiq.Explainer(
        model=model,
        data=X_train,          # 用训练集作为上下文
        labels=y_train,
        index="SV",            # Shapley values
        max_order=1,           # 一阶（单特征）
    )

    n_eval = min(EXPLAIN_MAX_EVAL, len(X_test))
    if n_eval <= 0:
        print("[warn] 测试样本不足，跳过 SHAP。")
        return

    X_explain = X_test[:n_eval]

    # 聚合 mean(|SHAP|) 所需
    abs_shap_sum = np.zeros(len(raw_cols), dtype=np.float64)

    # 可选：增量保存，避免长时间运行中断
    def save_partial(done):
        if done <= 0:
            return
        mean_abs_shap_partial = abs_shap_sum / done
        (pd.DataFrame({
            "feature": raw_cols,
            "mean_abs_shap_partial": mean_abs_shap_partial
        }).sort_values("mean_abs_shap_partial", ascending=False)
         .to_csv(OUT_CSV_PARTIAL, index=False, encoding="utf-8-sig"))

    print(f"[info] n_eval={n_eval}, budget={SHAP_BUDGET}")
    skipped = 0
    avg_t = None
    with tqdm(total=n_eval, desc="SHAP explaining", unit="sample", miniters=1, mininterval=0.1) as pbar:
        for i in range(n_eval):
            t1 = perf_counter()
            try:
                sv = explainer.explain(X_explain[i], budget=SHAP_BUDGET)
                # 将 dict 映射为长度为 n_features 的向量（只取单特征 Shapley）
                shap_vec = np.zeros(len(raw_cols), dtype=np.float64)
                for k, v in sv.dict_values.items():
                    if isinstance(k, tuple) and len(k) == 1:
                        shap_vec[k[0]] = v
                abs_shap_sum += np.abs(shap_vec)
            except ValueError as e:
                # 兼容“全常数将被删除”的错误：跳过该样本
                if "All features are constant" in str(e):
                    skipped += 1
                    print(f"[warn] 样本 {i+1}/{n_eval} 遇到常数子集，已跳过。", flush=True)
                else:
                    raise
            finally:
                dt = perf_counter() - t1
                avg_t = dt if avg_t is None else (0.9 * avg_t + 0.1 * dt)
                remain = (n_eval - (i + 1)) * (avg_t if avg_t else dt)
                pbar.update(1)
                pbar.set_postfix(last_s=f"{dt:.2f}", avg_s=f"{avg_t:.2f}", eta_s=f"{remain:.0f}")
                if SAVE_EVERY and (i + 1) % SAVE_EVERY == 0:
                    save_partial(i + 1 - skipped)

    # 汇总与保存最终结果（用有效样本数）
    effective = max(1, n_eval - skipped)
    mean_abs_shap = abs_shap_sum / effective
    imp_df = pd.DataFrame({"feature": raw_cols, "mean_abs_shap": mean_abs_shap})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print(f"\n[SHAP · RAW] Top-{TOPK_PRINT} features (mean |SHAP| over {effective} effective samples):")
    print(imp_df.head(TOPK_PRINT).to_string(index=False))

    imp_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[done] SHAP完成 · 用时 {(perf_counter()-t):.2f}s · 已保存：{os.path.abspath(OUT_CSV)}")

    # 总耗时
    section("全部完成")
    print(f"总用时：{(perf_counter()-t0):.2f}s")
    if os.path.exists(OUT_CSV_PARTIAL):
        print(f"[info] 中间结果（增量保存）在：{os.path.abspath(OUT_CSV_PARTIAL)}")


if __name__ == "__main__":
    main()



============ 数据读取与清洗 ============
[warn] 发现未映射到的标签值（将被剔除）：
Group
AD    219
CN    204
[hint] 如需包含它们，请调整 CLASSES 或预先过滤。
[info] 因标签未映射/缺失删除样本数：423
[info] 过滤零方差特征 2 列：['PXABNORM', 'NXABNORM']
[done] 用时 0.04s · 数据形状: X=(479, 137), y=479

============ 划分训练/测试集 ============
Train: (383, 137), Test: (96, 137), Pos rate(train)=0.329
[done] 用时 0.00s

============ 训练 TabPFN ============
[done] 用时 0.54s

============ 评估 TabPFN ============
[TabPFN] ACC=0.8125  AUC=0.8896
              precision    recall  f1-score   support

        SMCI       0.84      0.89      0.86        64
        PMCI       0.75      0.66      0.70        32

    accuracy                           0.81        96
   macro avg       0.79      0.77      0.78        96
weighted avg       0.81      0.81      0.81        96

[done] 用时 1.91s

============ SHAP（原始列 · TabPFNExplainer · 带进度） ============
[info] n_eval=64, budget=64


SHAP explaining: 100%|██████████| 64/64 [1:50:40<00:00, 103.76s/sample, avg_s=108.05, eta_s=0, last_s=101.93]   


[SHAP · RAW] Top-30 features (mean |SHAP| over 64 effective samples):
 feature  mean_abs_shap
PTETHCAT       4.774266
PTEDUCAT       4.246223
  PXNECK       3.526788
  FHQSIB       2.880508
BCCONSTP       2.839999
PXEXTREM       2.838016
 PTNOTRT       2.792623
  PXBACK       2.765609
BCSWEATN       2.757274
  PTHAND       2.737354
PXMUSCUL       2.606486
  FHQMOM       2.540719
BCDRYMTH       2.481202
 VSBPDIA       2.473067
 PTDOBYY       2.373890
 PTMARRY       2.371586
PXPERIPH       2.323116
 PTTLANG       2.320814
 PXOTHER       2.288104
VSTEMP-C       2.275544
 BCVOMIT       2.250925
 PXCHEST       2.149804
 PXABDOM       2.114022
 BCDIZZY       2.111947
PTRACCAT       2.108750
PXHEADEY       2.101976
BCDROWSY       2.099374
BCABDOMN       2.096638
 PXHEART       2.050665
BCDIARRH       2.006243
[done] SHAP完成 · 用时 6644.12s · 已保存：c:\Users\dongzj\Desktop\mmad\model_tabular\tabpfn_shap_importance_raw.csv

============ 全部完成 ============
总用时：6646.63s
[info] 中间结果（增量保存）在：c:\Users\dong

In [4]:
# -*- coding: utf-8 -*-
"""
TabPFN + SHAP(shapiq) · 原始列层面解释（带可视化进度与阶段耗时 + 稳健修复）
依赖：
    pip install tabpfn shapiq pandas scikit-learn numpy torch tqdm
"""

import os
import numpy as np
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from tqdm import tqdm

# --- TabPFN 兼容导入（官方包优先，其次 extensions） ---
try:
    from tabpfn import TabPFNClassifier  # 官方
except Exception:
    from tabpfn_extensions import TabPFNClassifier  # 你的 extensions 版本

import shapiq  # pip install shapiq


# ===================== 用户参数 =====================
CSV_PATH    = r"C:\Users\dongzj\Desktop\Multimodal_AD\adni_dataset\ADNI_Tabel.csv"
LABEL_COL   = "Group"
CLASSES     = ["SMCI", "PMCI"]       # 目标二分类标签
START_COL   = 4                      # 原始特征起始列（含）
TEST_SIZE   = 0.2
RANDOM_SEED = 42

# SHAP 参数（先小一点，确认流程 OK 再调大）
EXPLAIN_MAX_EVAL = 32               # 从测试集中抽多少样本做解释
SHAP_BUDGET      = 32               # Shapley 预算（子集采样预算，越大越稳越慢）
TOPK_PRINT       = 30
OUT_CSV          = "tabpfn_shap_importance_raw.csv"
OUT_CSV_PARTIAL  = "tabpfn_shap_importance_raw_partial.csv"  # 增量保存文件
SAVE_EVERY       = 32                # 每处理多少样本增量保存一次

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def section(msg: str):
    print("\n" + "=" * 12 + f" {msg} " + "=" * 12, flush=True)


def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    t0 = perf_counter()

    # ===================== 读取&清洗 =====================
    section("数据读取与清洗")
    t = perf_counter()
    df = pd.read_csv(CSV_PATH)

    # 原始特征列
    raw_cols = list(df.columns[START_COL:])

    # 标签清洗：统一去空格
    labels_raw = df[LABEL_COL].astype(str).str.strip()

    # 映射
    mapping = {CLASSES[0]: 0, CLASSES[1]: 1}
    y_map = labels_raw.map(mapping)

    # 二次“大小写无关”兜底
    if y_map.isna().any():
        lower_map = {CLASSES[0].lower(): 0, CLASSES[1].lower(): 1}
        y_map2 = labels_raw.str.lower().map(lower_map)
        y_map = y_map.fillna(y_map2)

    # 未映射的提示并剔除
    if y_map.isna().any():
        unmapped_vals = labels_raw[y_map.isna()].value_counts()
        print("[warn] 发现未映射到的标签值（将被剔除）：")
        print(unmapped_vals.to_string())
        print("[hint] 如需包含它们，请调整 CLASSES 或预先过滤。")

    keep_mask = ~y_map.isna()
    dropped = (~keep_mask).sum()
    if dropped > 0:
        print(f"[info] 因标签未映射/缺失删除样本数：{dropped}")
    df = df.loc[keep_mask].reset_index(drop=True)
    y_all = y_map.loc[keep_mask].astype(int).values

    # 若只剩一个类别，直接报错提示
    if np.unique(y_all).size < 2:
        raise ValueError(f"仅剩单一类别：{np.unique(y_all)}。请检查 LABEL_COL/CLASSES 或过滤策略。")

    # 特征清洗：inf/-inf -> NaN -> 中位数插补
    X_df = df[raw_cols].copy()
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="median")
    X_all = imputer.fit_transform(X_df.values).astype(np.float32)

    # === 新增：过滤零方差特征（全局）===
    rng = np.ptp(X_all, axis=0)  # max-min
    nonconst_mask = rng > 1e-12  # 或用 X_all.std(axis=0) > 1e-12
    if nonconst_mask.sum() < len(nonconst_mask):
        removed = [c for c, keep in zip(raw_cols, nonconst_mask) if not keep]
        print(f"[info] 过滤零方差特征 {len(removed)} 列：{removed[:10]}{' ...' if len(removed)>10 else ''}")
    X_all = X_all[:, nonconst_mask]
    raw_cols = [c for c, keep in zip(raw_cols, nonconst_mask) if keep]

    # 断言无 NaN/Inf
    if not np.isfinite(X_all).all():
        raise ValueError("插补后仍存在 NaN/Inf，请检查数据。")

    print(f"[done] 用时 {(perf_counter()-t):.2f}s · 数据形状: X={X_all.shape}, y={y_all.shape[0]}")

    # ===================== 划分 =====================
    section("划分训练/测试集")
    t = perf_counter()
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
    )
    print(f"Train: {X_train.shape}, Test: {X_test.shape}, Pos rate(train)={y_train.mean():.3f}")
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # ===================== 训练 TabPFN =====================
    section("训练 TabPFN")
    t = perf_counter()
    model = TabPFNClassifier(device=DEVICE)
    model.fit(X_train, y_train)
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # 评估
    section("评估 TabPFN")
    t = perf_counter()
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"[TabPFN] ACC={acc:.4f}  AUC={auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASSES))
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # ===================== SHAP（原始列层面，带进度条） =====================
    section("SHAP（原始列 · TabPFNExplainer · 带进度）")
    t = perf_counter()
    explainer = shapiq.Explainer(
        model=model,
        data=X_train,          # 用训练集作为上下文
        labels=y_train,
        index="SV",            # Shapley values
        max_order=1,           # 一阶（单特征）
    )

    n_eval = min(EXPLAIN_MAX_EVAL, len(X_test))
    if n_eval <= 0:
        print("[warn] 测试样本不足，跳过 SHAP。")
        return

    X_explain = X_test[:n_eval]

    # 聚合 mean(|SHAP|) 所需
    abs_shap_sum = np.zeros(len(raw_cols), dtype=np.float64)

    # 可选：增量保存，避免长时间运行中断
    def save_partial(done):
        if done <= 0:
            return
        mean_abs_shap_partial = abs_shap_sum / done
        (pd.DataFrame({
            "feature": raw_cols,
            "mean_abs_shap_partial": mean_abs_shap_partial
        }).sort_values("mean_abs_shap_partial", ascending=False)
         .to_csv(OUT_CSV_PARTIAL, index=False, encoding="utf-8-sig"))

    print(f"[info] n_eval={n_eval}, budget={SHAP_BUDGET}")
    skipped = 0
    avg_t = None
    with tqdm(total=n_eval, desc="SHAP explaining", unit="sample", miniters=1, mininterval=0.1) as pbar:
        for i in range(n_eval):
            t1 = perf_counter()
            try:
                sv = explainer.explain(X_explain[i], budget=SHAP_BUDGET)
                # 将 dict 映射为长度为 n_features 的向量（只取单特征 Shapley）
                shap_vec = np.zeros(len(raw_cols), dtype=np.float64)
                for k, v in sv.dict_values.items():
                    if isinstance(k, tuple) and len(k) == 1:
                        shap_vec[k[0]] = v
                abs_shap_sum += np.abs(shap_vec)
            except ValueError as e:
                # 兼容“全常数将被删除”的错误：跳过该样本
                if "All features are constant" in str(e):
                    skipped += 1
                    print(f"[warn] 样本 {i+1}/{n_eval} 遇到常数子集，已跳过。", flush=True)
                else:
                    raise
            finally:
                dt = perf_counter() - t1
                avg_t = dt if avg_t is None else (0.9 * avg_t + 0.1 * dt)
                remain = (n_eval - (i + 1)) * (avg_t if avg_t else dt)
                pbar.update(1)
                pbar.set_postfix(last_s=f"{dt:.2f}", avg_s=f"{avg_t:.2f}", eta_s=f"{remain:.0f}")
                if SAVE_EVERY and (i + 1) % SAVE_EVERY == 0:
                    save_partial(i + 1 - skipped)

    # 汇总与保存最终结果（用有效样本数）
    effective = max(1, n_eval - skipped)
    mean_abs_shap = abs_shap_sum / effective
    imp_df = pd.DataFrame({"feature": raw_cols, "mean_abs_shap": mean_abs_shap})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print(f"\n[SHAP · RAW] Top-{TOPK_PRINT} features (mean |SHAP| over {effective} effective samples):")
    print(imp_df.head(TOPK_PRINT).to_string(index=False))

    imp_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[done] SHAP完成 · 用时 {(perf_counter()-t):.2f}s · 已保存：{os.path.abspath(OUT_CSV)}")

    # 总耗时
    section("全部完成")
    print(f"总用时：{(perf_counter()-t0):.2f}s")
    if os.path.exists(OUT_CSV_PARTIAL):
        print(f"[info] 中间结果（增量保存）在：{os.path.abspath(OUT_CSV_PARTIAL)}")


if __name__ == "__main__":
    main()



============ 数据读取与清洗 ============
[warn] 发现未映射到的标签值（将被剔除）：
Group
AD    219
CN    204
[hint] 如需包含它们，请调整 CLASSES 或预先过滤。
[info] 因标签未映射/缺失删除样本数：423
[info] 过滤零方差特征 2 列：['PXABNORM', 'NXABNORM']
[done] 用时 0.06s · 数据形状: X=(479, 137), y=479

============ 划分训练/测试集 ============
Train: (383, 137), Test: (96, 137), Pos rate(train)=0.329
[done] 用时 0.01s

============ 训练 TabPFN ============
[done] 用时 0.74s

============ 评估 TabPFN ============
[TabPFN] ACC=0.8125  AUC=0.8896
              precision    recall  f1-score   support

        SMCI       0.84      0.89      0.86        64
        PMCI       0.75      0.66      0.70        32

    accuracy                           0.81        96
   macro avg       0.79      0.77      0.78        96
weighted avg       0.81      0.81      0.81        96

[done] 用时 1.97s

============ SHAP（原始列 · TabPFNExplainer · 带进度） ============
[info] n_eval=32, budget=32


SHAP explaining: 100%|██████████| 32/32 [20:04<00:00, 37.65s/sample, avg_s=34.50, eta_s=0, last_s=34.20]   


[SHAP · RAW] Top-30 features (mean |SHAP| over 32 effective samples):
     feature  mean_abs_shap
      ADAS13   5.172989e+34
        MMSE   5.172989e+34
      FHQMOM   4.326779e+24
     PTMARRY   4.011981e+24
      PXNECK   2.754286e+24
      VSRESP   2.754252e+24
     VSPULSE   1.886579e+24
    PXHEADEY   1.810600e+24
      FHQSIB   1.181761e+24
    VSTEMP-C   1.181750e+24
     VSBPSYS   6.288843e+23
    PTRACCAT   6.288603e+23
      PTHOME   3.147885e+23
    PTEDUCAT   7.598851e+22
     BCOTHER   4.896144e+19
    MH17MALI   4.814428e+19
     BCCHEST   3.162054e+19
    BCCRYING   2.970265e+19
     VSBPDIA   2.177580e+19
    PXGENAPP   1.945189e+19
    PXPERIPH   1.681382e+19
    FHQDADAD   1.659859e+19
    BCPALPIT   1.570073e+19
MPACCTRAILSB   1.477881e+19
    BCVISION   1.268011e+19
     MHPSYCH   1.246749e+19
    BCINSOMN   1.163779e+19
     BCDIZZY   1.109267e+19
    MH8MUSCL   1.094134e+19
     PTDOBYY   1.011804e+19
[done] SHAP完成 · 用时 1209.44s · 已保存：c:\Users\dongzj\Desktop\mma

In [ ]:
# -*- coding: utf-8 -*-
"""
绘制 SHAP Top-20 复合图（每个模型一列：左侧 bar + 右侧 beeswarm）
输入：长表 CSV，列包含 ['model','feature','shap','value']；若无 shap/value 的逐样本数据，则只画 bar。
输出：shap_summary_top20.png
"""
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import Normalize

# ========= 用户配置 =========
CSV_PATH     = "shap_results_long.csv"   # 这里替换成你的结果表格路径
TOPK         = 20
OUT_FIG      = "shap_summary_top20.png"
FIG_W, FIG_H = 16, 5.2                   # 画布大小（宽×高，英寸）
DPI          = 200

# ========== 读取 ==========
df = pd.read_csv(CSV_PATH)

# 兼容“只有全局重要性表”的情况：至少需要 feature / mean_abs_shap
has_long = set(["model","feature","shap","value"]).issubset(df.columns)
if not has_long:
    # 期望列名：feature, mean_abs_shap，（可选）model
    assert "feature" in df.columns, "CSV 至少需要包含 feature 列"
    assert "mean_abs_shap" in df.columns, "只有全局重要性时需要 mean_abs_shap 列"
    if "model" not in df.columns:
        df["model"] = "Model"
    # 用伪长表以便后续处理
    tmp = df.copy()
    tmp["shap"] = np.nan
    tmp["value"] = np.nan
    df = tmp[["model","feature","shap","value","mean_abs_shap"]]

models = list(df["model"].dropna().astype(str).unique())
models.sort()   # 固定顺序（你也可以自定义）

# ========== 计算各模型 Top-K ==========
topk_by_model = {}
for m in models:
    sub = df[df["model"] == m]
    if has_long:
        # 先按 feature 聚合 mean(|shap|)
        g = sub.groupby("feature")["shap"].apply(lambda x: np.mean(np.abs(x.dropna()))).reset_index(name="mean_abs_shap")
    else:
        g = sub[["feature","mean_abs_shap"]].drop_duplicates()
    g = g.sort_values("mean_abs_shap", ascending=False).head(TOPK)
    # 为了 y 轴从上到下显示：反转顺序
    g = g.iloc[::-1].reset_index(drop=True)
    topk_by_model[m] = g

# ========== 画图：每个模型一列（左 bar + 右 beeswarm）==========
n_cols = len(models)
fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI)
outer = gridspec.GridSpec(1, n_cols, wspace=0.25)  # 每个模型占一列

for col_idx, m in enumerate(models):
    g = topk_by_model[m]
    feats_order = g["feature"].tolist()        # 自上而下的顺序
    means = g["mean_abs_shap"].values

    # 内部再切两个子区：左侧bar / 右侧beeswarm
    inner = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[col_idx], wspace=0.05, width_ratios=[1.0, 3.0])

    # ----- 左：bar（水平）-----
    ax_bar = plt.Subplot(fig, inner[0])
    y_pos = np.arange(len(feats_order))
    ax_bar.barh(y_pos, means, align="center")
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(feats_order, fontsize=9)
    ax_bar.invert_yaxis()  # 让 Top 的在上方
    ax_bar.set_xlabel("Mean(|SHAP|)")
    ax_bar.set_xlim(left=0)
    ax_bar.grid(axis="x", linestyle="--", alpha=0.4)
    ax_bar.set_title(m, fontsize=11, loc="left")
    fig.add_subplot(ax_bar)

    # ----- 右：beeswarm（需要逐样本 shap/value）-----
    ax_swarm = plt.Subplot(fig, inner[1])
    if has_long:
        sub = df[(df["model"] == m) & (df["feature"].isin(feats_order))].copy()
        # 按 feats_order 设定 y 的整数编码
        f2y = {f:i for i, f in enumerate(feats_order)}
        sub["y"] = sub["feature"].map(f2y)

        # 颜色映射：按每个特征的值归一化（和 SHAP 官方类似）
        # 为了稳健，这里分特征分别归一化，再拼起来
        colors = np.zeros(len(sub))
        for f in feats_order:
            mask = (sub["feature"] == f)
            vals = sub.loc[mask, "value"].astype(float)
            if vals.notna().sum() > 0:
                vmin, vmax = np.nanmin(vals), np.nanmax(vals)
                if math.isclose(vmin, vmax):
                    normed = np.full(vals.shape, 0.5)  # 常数列
                else:
                    normed = (vals - vmin) / (vmax - vmin)
                colors[mask.values] = normed
            else:
                colors[mask.values] = 0.5
        # 抖动（让同一 y 的点分散开）
        # 依据 y 每行的样本数做轻微左右抖动
        rng = np.random.default_rng(123)
        jitter = (rng.random(len(sub)) - 0.5) * 0.15

        # 画散点：x=shap, y=feature 编码 + 抖动，颜色=归一化value（cmap 默认）
        ax_swarm.scatter(sub["shap"].values,
                         sub["y"].values + jitter,
                         c=colors, cmap=plt.cm.get_cmap(), s=6, alpha=0.8, linewidths=0)

        # 辅助线
        ax_swarm.axvline(0.0, color="k", linewidth=0.8, alpha=0.6)
        ax_swarm.set_yticks(y_pos)
        ax_swarm.set_yticklabels([])  # 右侧不重复显示 ytick label
        ax_swarm.set_xlabel("SHAP value (impact on model output)")
        ax_swarm.grid(axis="x", linestyle="--", alpha=0.25)
    else:
        # 没有逐样本 shap/value，就给出提示文本
        ax_swarm.text(0.5, 0.5, "No per-sample SHAP values.\nOnly bar is drawn.",
                      ha="center", va="center", transform=ax_swarm.transAxes)
        ax_swarm.set_xticks([])
        ax_swarm.set_yticks([])

    fig.add_subplot(ax_swarm)

# 整体标题与收尾
fig.suptitle("Top-20 features by Mean(|SHAP|) with beeswarm per model", y=0.98, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUT_FIG, dpi=DPI, bbox_inches="tight")
print(f"[saved] {os.path.abspath(OUT_FIG)}")
